# 单日订单分配、仓库扩容与中断分析
使用 **Python (gurobi-env)** 内核，从上到下运行。输入文件为 `data/Supply chain logisitcs problem.xlsx`。金额单位为美元，容量单位为笔/天，当日处理指安排发运。

| 参数 | 从哪里取得 |
|---|---|
| 订单集合 $I$、件数 $q_i$、重量 $g_i$ | `OrderList` 中 DTD/DTP 订单及其数量、原始重量。所有订单在规划日开始时可处理。 |
| 仓库容量 $B_w$、单件仓储成本 $h_w$ | `WhCapacities` 的日订单数、`WhCosts` 的 Cost/unit。 |
| 可选方案 $A_i$ | 用 `ProductsPerPlant`、`VmiCustomers`、`PlantPorts` 筛选仓库和港口，再匹配 `FreightRates` 的目的港、原服务类别和重量区间。方案 $a$ 记录仓库、出发港与报价行号。 |
| 方案成本 $c_{ia}$ | 三个问题均逐单计费。仓储费用为 $q_i h_w$。空运费用取最低收费与“原始重量 × 费率”的较大值，地运费用取该档单独用车报价。 |
| 预留容量 $R_w$、扩容幅度 $\Delta_w$、扩容名额 $K$ | 基准 $R_w=0$。预留情景按历史仓库统计 CRF 笔数。扩容情景取 $\Delta_w=100$ 笔/天、$K=2$ 座。 |

[原始数据说明](https://doi.org/10.17633/rd.brunel.7558679.v2) · [运费计算依据，第 4.2 节](https://bura.brunel.ac.uk/bitstream/2438/20185/5/Preprint.pdf)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display

filename = "Supply chain logisitcs problem.xlsx"
paths = [Path("data") / filename, Path("group_project/data") / filename, Path.home() / "Downloads" / filename]
workbook = next(p for p in paths if p.is_file())
sheets = pd.read_excel(workbook, sheet_name=None, engine="openpyxl")
for sheet in sheets.values():
    sheet.columns = sheet.columns.str.strip()

orders_all = sheets["OrderList"].rename(columns={
    "Order ID": "order", "Product ID": "product", "Customer": "customer",
    "Plant Code": "historical_plant", "Carrier": "historical_carrier", "Destination Port": "destination",
    "Service Level": "service", "Unit quantity": "units", "Weight": "weight", "TPT": "historical_days"
})
orders_all["order"] = orders_all["order"].astype(str)
assert orders_all["order"].is_unique
orders = orders_all.loc[orders_all.service.isin(["DTD", "DTP"])].copy()
capacity = sheets["WhCapacities"].set_index("Plant ID")["Daily Capacity"].sort_index().astype(int)
storage = sheets["WhCosts"].set_index("WH")["Cost/unit"]
crf_count = (orders_all.loc[orders_all.service.eq("CRF"), "historical_plant"]
             .value_counts().reindex(capacity.index, fill_value=0))
assert (crf_count <= capacity).all()

products = sheets["ProductsPerPlant"].rename(columns={"Plant Code": "plant", "Product ID": "product"})
ports = sheets["PlantPorts"].rename(columns={"Plant Code": "plant", "Port": "port"})
vmi = sheets["VmiCustomers"].rename(columns={"Plant Code": "plant", "Customers": "customer"})
rates = sheets["FreightRates"].drop_duplicates().rename(columns={
    "Carrier": "carrier", "orig_port_cd": "port", "dest_port_cd": "destination", "svc_cd": "service",
    "minm_wgh_qty": "low", "max_wgh_qty": "high", "minimum cost": "minimum",
    "mode_dsc": "mode", "tpt_day_cnt": "days"
}).copy()
rates["quote"] = rates.index + 2  # 原 Excel 行号，含表头
rates["mode"] = rates["mode"].str.strip()
assert set(rates["mode"]) == {"AIR", "GROUND"}

# 名单约束从仓库出发：有名单的仓库只能服务列明的客户。
routes = (orders[["order", "product", "customer", "destination", "service", "units", "weight", "historical_days"]]
          .merge(products, on="product").merge(ports, on="plant")
          .merge(rates, on=["port", "destination", "service"]))
routes = routes.loc[routes.weight.between(routes.low, routes.high, inclusive="both")].copy()
allowed_pairs = pd.MultiIndex.from_frame(vmi[["plant", "customer"]])
routes["vmi_ok"] = (~routes.plant.isin(vmi.plant)
                    | pd.MultiIndex.from_frame(routes[["plant", "customer"]]).isin(allowed_pairs))
routes["warehouse_cost"] = routes.units * routes.plant.map(storage)
routes["freight"] = np.where(routes["mode"].eq("AIR"),
                             np.maximum(routes.minimum, routes.weight * routes.rate), routes.rate)
routes["cost"] = routes.warehouse_cost + routes.freight
assert routes["cost"].notna().all()
# 保留完整报价表。每次应用情景限制之后，才压缩同订单、同仓库的等价选择。
routes = routes.sort_values(["cost", "days", "quote", "plant"], kind="stable")
eligible = routes.loc[routes.vmi_ok]
def lowest_by_plant(options):
    return options.drop_duplicates(["order", "plant"]).reset_index(drop=True)

base_options = lowest_by_plant(eligible)
display(pd.DataFrame([{
    "DTD/DTP 订单": len(orders), "有候选方案": eligible.order.nunique(),
    "原始报价": len(sheets["FreightRates"]), "去重后报价": len(rates),
    "零重量订单": orders.weight.eq(0).sum(), "订单与仓库组合": len(base_options)
}]))

,DTD/DTP 订单,有候选方案,原始报价,去重后报价,零重量订单,订单与仓库组合
0,8361,8361,1540,1537,2,10046


## 问题一　现有仓容一天能处理多少单，最低成本是多少？

**模型**　先最大流，再在最大流量下求最小费用流。$x_{ia}$ 表示订单 $i$ 是否选用方案 $a$。每单消耗一个名额，整数容量下存在整单最优解，用连续变量和单纯形法求解。

$$\sum_{a\in A_i}x_{ia}\le 1,\qquad
\sum_{i,a:\,w(a)=w} x_{ia}\le B_w-R_w,\qquad x_{ia}\ge 0.$$

第一步最大化 $N=\sum_{i,a}x_{ia}$。第二步固定 $N=N^*$，最小化 $\sum_{i,a}c_{ia}x_{ia}$。

In [2]:
env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.setParam("Method", 1)  # 对偶单纯形法
env.setParam("Threads", 2)
env.setParam("MIPGap", 0)
env.setParam("MIPGapAbs", 0)
env.start()

def add_assignment(model, options, caps, *, required=False, name="assign"):
    # 每个“订单、仓库”已有一个最低价方案，固定清单时要求每个编号恰好分配一次。
    x = model.addVars(list(zip(options.order, options.plant)), ub=1, name=name)
    for i in options.order.unique():
        count_i = x.sum(i, "*")
        model.addConstr(count_i == 1 if required else count_i <= 1)
    limits = model.addConstrs((x.sum("*", w) <= caps[w] for w in caps), name=name + "_capacity")
    cost = gp.LinExpr(options.cost.to_list(), list(x.values()))
    return x, cost, limits

def optimum(model, objective, sense):
    model.setObjective(objective, sense)
    model.optimize()
    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Gurobi 求解状态 {model.Status}")
    return model.ObjVal

def solve_assignment(options, caps, *, throughput_only=False):
    if (caps < 0).any():
        raise ValueError("可用仓容不能为负数")
    with gp.Model(env=env) as model:
        x, cost, limits = add_assignment(model, options, caps.to_dict())
        count = x.sum()
        maximum = round(optimum(model, count, GRB.MAXIMIZE))
        if throughput_only:
            return {"maximum": maximum}
        throughput_dual = pd.Series(model.getAttr("Pi", limits))
        model.addConstr(count == maximum)
        minimum = optimum(model, cost, GRB.MINIMIZE)
        values = np.array(model.getAttr("X", list(x.values())))
        assert np.allclose(values, values.round(), atol=1e-6, rtol=0)
        chosen = options.loc[values > 0.5].copy()
        load = chosen.plant.value_counts().reindex(caps.index, fill_value=0)
        assert len(chosen) == maximum and chosen.order.is_unique and (load <= caps).all()
        assert np.isclose(chosen.cost.sum(), minimum, atol=1e-5, rtol=0)
        return {"maximum": maximum, "cost": minimum, "chosen": chosen, "load": load,
                "throughput_dual": throughput_dual, "cost_dual": pd.Series(model.getAttr("Pi", limits))}


In [3]:
base = solve_assignment(base_options, capacity)
without_vmi = solve_assignment(lowest_by_plant(routes), capacity)
assert (len(orders), eligible.order.nunique(), base["maximum"], without_vmi["maximum"]) == (8361, 8361, 2291, 2636)
assert np.isclose(base["cost"], 743755.6256404707, atol=0.01, rtol=0)

def baseline_row(label, demand, options, result):
    return {"情景": label, "订单池": demand, "有候选方案": options.order.nunique(),
            "最多处理": result["maximum"], "最低运营成本（美元）": round(result["cost"], 2)}

checks = [baseline_row("基准", len(orders), base_options, base),
          baseline_row("放宽 VMI", len(orders), routes, without_vmi)]
for label, demand, options, caps in [
    ("运输天数不超过历史值", len(orders), eligible.loc[eligible.days <= eligible.historical_days], capacity),
    ("剔除零重量订单", int(orders.weight.gt(0).sum()), eligible.loc[eligible.weight > 0], capacity),
    ("按历史仓库预留 CRF 名额", len(orders), eligible, capacity - crf_count)
]:
    result = solve_assignment(lowest_by_plant(options), caps)
    checks.append(baseline_row(label, demand, options, result))
baseline_checks = pd.DataFrame(checks).set_index("情景")
display(baseline_checks)
order_plan = base["chosen"][["order", "plant", "port", "carrier", "quote", "days", "warehouse_cost", "freight", "cost"]]
display(order_plan.head(3).round(2).rename(columns={"order":"订单编号","plant":"仓库","port":"港口","carrier":"承运商",
                                                    "quote":"报价行","days":"运输天数","warehouse_cost":"仓储费",
                                                    "freight":"运费","cost":"总成本"}))
print(f"基准可处理 {base['maximum']:,} 笔，最低成本为 {base['cost']:,.2f} 美元；完整逐单方案保存在 order_plan。")

,订单池,有候选方案,最多处理,最低运营成本（美元）
情景,,,,
基准,8361,8361,2291,743755.63
放宽 VMI,8361,8361,2636,805806.24
运输天数不超过历史值,8361,8361,2277,730919.36
剔除零重量订单,8359,8359,2290,743600.70
按历史仓库预留 CRF 名额,8361,8361,1578,637347.78


,订单编号,仓库,港口,承运商,报价行,运输天数,仓储费,运费,总成本
0,1447368565.7,PLANT04,PORT05,V444_5,210,3,106.27,5.81,112.08
1,1447161166.7,PLANT04,PORT05,V444_5,210,3,107.13,5.81,112.93
2,1447220210.7,PLANT04,PORT05,V444_5,210,3,107.55,5.81,113.36


基准可处理 2,291 笔，最低成本为 743,755.63 美元；完整逐单方案保存在 order_plan。


## 问题二　给哪些仓库扩容，每天能多处理多少笔，同一批订单能省多少钱？

每座入选仓库每天增加 100 个名额，最多选两座。$y_w=1$ 表示仓库 $w$ 扩容。基准预留量为零，扩容后该仓日容量为 $B_w+100y_w$，且 $\sum_w y_w\le 2$。

- **新增处理量**　用全部待处理订单分别求出扩容前后的单日最大处理笔数，取两者之差。
- **运营节省**　固定问题一基准方案选中的全部订单编号。在两种容量下分别完成这批订单，允许调整仓库和发运路线。用扩容前的最低成本减去扩容后的最低成本。

先逐仓试算，再用混合整数线性规划选择组合。先选出最大处理量最高的组合。如有并列，选完成固定清单成本最低的组合。

In [4]:
EXTRA = 100
SITE_LIMIT = 2
N0 = base["maximum"]
base_ids = frozenset(order_plan.order)
fixed_options = base_options.loc[base_options.order.isin(base_ids)]
assert len(base_ids) == fixed_options.order.nunique() == N0

def fixed_batch_cost(caps):
    result = solve_assignment(fixed_options, caps)
    assert frozenset(result["chosen"].order) == base_ids
    return result["cost"]

before_cost = fixed_batch_cost(capacity)
assert np.isclose(before_cost, base["cost"], atol=1e-5, rtol=0)

single_rows = []
for plant in capacity.index:
    caps = capacity.copy()
    caps.loc[plant] += EXTRA
    maximum = solve_assignment(base_options, caps, throughput_only=True)["maximum"]
    saving = before_cost - fixed_batch_cost(caps)
    assert maximum >= N0 and saving >= -1e-5
    single_rows.append({"仓库": plant, "原容量": capacity[plant], "基准负载": base["load"][plant],
                        "每日新增处理笔数": maximum - N0, "固定订单节省（美元）": max(0.0, saving)})
single_expansion = (pd.DataFrame(single_rows).set_index("仓库")
                    .sort_values(["每日新增处理笔数", "固定订单节省（美元）"], ascending=False, kind="stable"))
display(single_expansion.style.format({"固定订单节省（美元）": "{:,.2f}"}))


,原容量,基准负载,每日新增处理笔数,固定订单节省（美元）
仓库,,,,
PLANT02,138,138,100,126.93
PLANT03,1013,1013,100,0.00
PLANT12,209,209,65,"8,999.06"
PLANT10,118,118,65,0.00
PLANT09,11,11,43,0.00
PLANT01,1070,0,0,0.00
PLANT04,554,375,0,0.00
PLANT05,385,179,0,0.00
PLANT06,49,0,0,0.00


In [5]:
# 两次对照共用扩容地点，各自在扩容后的仓容内分单。
with gp.Model(env=env) as model:
    y = model.addVars(capacity.index.to_list(), vtype=GRB.BINARY, name="expand")
    model.addConstr(y.sum() <= SITE_LIMIT)
    caps_y = {w: int(capacity[w]) + EXTRA * y[w] for w in capacity.index}

    x, _, _ = add_assignment(model, base_options, caps_y)
    maximum = round(optimum(model, x.sum(), GRB.MAXIMIZE))
    model.addConstr(x.sum() == maximum)
    _, fixed_cost, _ = add_assignment(model, fixed_options, caps_y, required=True, name="fixed")
    minimum = optimum(model, fixed_cost, GRB.MINIMIZE)
    sites = sorted(w for w in capacity.index if y[w].X > 0.5)

expanded_capacity = capacity.copy()
expanded_capacity.loc[sites] += EXTRA
expanded = solve_assignment(base_options, expanded_capacity, throughput_only=True)
after_cost = fixed_batch_cost(expanded_capacity)
assert expanded["maximum"] == maximum and np.isclose(after_cost, minimum, atol=1e-5, rtol=0)
gain = maximum - N0
saving = before_cost - after_cost  # 先求差，最后显示到美分
sites_text = "、".join(sites)
comparison = pd.DataFrame([
    ["原容量", N0, 0, before_cost, 0.0],
    [sites_text + " 扩容", maximum, gain, after_cost, saving]
], columns=["容量方案", "单日最多处理笔数", "每日新增处理笔数",
            "固定订单最低成本（美元）", "固定订单节省（美元）"]).set_index("容量方案")
display(comparison.style.format({"固定订单最低成本（美元）": "{:,.2f}", "固定订单节省（美元）": "{:,.2f}"}))
print(f"{sites_text} 各增加 {EXTRA} 个名额后，每天最多多处理 {gain:,} 笔。"
      f"单独比较基准的 {N0:,} 笔订单，全部完成可节省 {saving:,.2f} 美元。")


,单日最多处理笔数,每日新增处理笔数,固定订单最低成本（美元）,固定订单节省（美元）
容量方案,,,,
原容量,2291,0,"743,755.63",0.00
PLANT02、PLANT03 扩容,2491,200,"743,628.69",126.93


PLANT02、PLANT03 各增加 100 个名额后，每天最多多处理 200 笔。单独比较基准的 2,291 笔订单，全部完成可节省 126.93 美元。


## 问题三　停运影响最大的是哪里，应提前给哪些仓库扩容？

### 3.1　原容量下，哪里停运会损失最多订单？

逐一模拟一座仓库或一家承运商停运。用第一问的订单池重新分配，按“正常日最大处理量 − 停运后最大处理量”衡量损失。

In [6]:
plants = capacity.index.to_list()
carriers = sorted(rates.carrier.unique())

# 仓库停运移除该仓的订单选项。承运商停运先移除其报价，再查找剩余报价。
scenario_options = {"正常运营": base_options}
for plant in plants:
    scenario_options[plant] = base_options.loc[base_options.plant != plant]
for carrier in carriers:
    scenario_options[carrier] = lowest_by_plant(eligible.loc[eligible.carrier != carrier])

outage_rows = []
for kind, nodes in [("仓库", plants), ("承运商", carriers)]:
    for node in nodes:
        served = solve_assignment(scenario_options[node], capacity, throughput_only=True)["maximum"]
        outage_rows.append({"类型": kind, "停运对象": node, "停运后处理笔数": served,
                            "损失笔数": N0 - served, "损失占比（%）": 100 * (N0 - served) / N0})

disruptions = pd.DataFrame(outage_rows).sort_values(["损失笔数", "停运对象"], ascending=[False, True])
print(f"原容量正常日最多处理 {N0:,} 笔。{len(disruptions)} 个单一停运情景中，处理量损失最大的是：")
display(disruptions.head(4).set_index(["类型", "停运对象"]).round(1))

原容量正常日最多处理 2,291 笔。28 个单一停运情景中，处理量损失最大的是：


停运后处理笔数  损失笔数  损失占比（%）
类型  停运对象                           
仓库  PLANT03     1278  1013     44.2
承运商 V444_5      1790   501     21.9
仓库  PLANT04     1916   375     16.4
    PLANT12     2082   209      9.1

### 3.2　考虑停运风险，怎样扩容才能让期望处理量最大？

事先选定最多两座仓库，每座增加 100 笔/天。每个候选方案都经历相同的正常日和 28 种单一停运情景，再按当天可用的网络分配订单。

数据未给出停运频率。假设发生停运时，28 种情景等可能。设 $p$ 为当天发生停运的概率，则单日期望处理量为 $(1-p)\times\text{正常日处理量}+p\times\text{停运日平均处理量}$。

In [7]:
from itertools import combinations

choices = [sites_to_expand for k in range(SITE_LIMIT + 1) for sites_to_expand in combinations(plants, k)]
plan_caps = pd.DataFrame([capacity + EXTRA * capacity.index.isin(choice) for choice in choices],
                         index=["、".join(choice) or "原容量" for choice in choices], columns=plants)
plan_caps.index.name = "扩容仓库"

def evaluate_capacity_plans(options, plans):
    # 每笔只占一个名额。可选仓库完全相同的订单可合并计数，保留相同的最大处理量。
    groups = options.groupby("order", sort=False)["plant"].agg(lambda ws: tuple(sorted(ws))).value_counts()
    with gp.Model(env=env) as model:
        incoming = {w: [] for w in plans.columns}
        for warehouses, n_orders in groups.items():
            flow = model.addVars(warehouses, obj=1)
            model.addConstr(flow.sum() <= int(n_orders))
            for w in warehouses:
                incoming[w].append(flow[w])
        limits = [model.addConstr(gp.quicksum(incoming[w]) <= 0) for w in plans.columns]
        model.ModelSense = GRB.MAXIMIZE
        totals = []
        for rhs in plans.itertuples(index=False, name=None):
            model.setAttr("RHS", limits, rhs)
            model.optimize()
            if model.Status != GRB.OPTIMAL:
                raise RuntimeError(f"最大处理量求解状态 {model.Status}")
            totals.append(round(model.ObjVal))
    return totals

# 完整矩阵：每行一个事先选定的扩容方案，每列一个运营情景。
plan_throughput = pd.DataFrame({name: evaluate_capacity_plans(options, plan_caps)
                                for name, options in scenario_options.items()}, index=plan_caps.index)
outage_names = disruptions["停运对象"].to_list()
assert plan_throughput.at["原容量", "正常运营"] == N0
assert plan_throughput.loc["原容量", outage_names].to_list() == disruptions["停运后处理笔数"].to_list()
assert plan_throughput.at[sites_text, "正常运营"] == expanded["maximum"]

In [8]:
normal_day = plan_throughput["正常运营"]
outage_mean = plan_throughput[outage_names].mean(axis=1)

def expected_orders(p_outage):
    """按给定的单日停运概率，返回所有方案的期望处理量。"""
    if not 0 <= p_outage <= 1:
        raise ValueError("停运概率必须在 0 到 1 之间")
    return (1 - p_outage) * normal_day + p_outage * outage_mean

risk_comparison = pd.DataFrame({"正常日处理笔数": normal_day, "停运日平均处理笔数": expected_orders(1)})
risk_comparison["较原容量的停运日平均增量"] = outage_mean - outage_mean["原容量"]
ranked = risk_comparison.sort_values(["停运日平均处理笔数", "正常日处理笔数"], ascending=False)
print(f"共比较 {len(plan_caps)} 个方案。下表列出原容量与停运日平均处理量最高的三个扩容方案。")
display(pd.concat([ranked.loc[["原容量"]], ranked.drop(index="原容量").head(3)]).round(2))

# 期望值是两列处理量的加权平均。两列同时居首的方案对任意停运概率都最优。
best_in_both = normal_day.index[normal_day.eq(normal_day.max()) & outage_mean.eq(outage_mean.max())]
if len(best_in_both):
    for plan in best_in_both:
        print(f"{plan} 在正常日和停运日平均都居首：{normal_day[plan]:,} 笔和 {outage_mean[plan]:,.2f} 笔。"
              "在各停运情景等可能的假设下，任意单日停运概率都可选这个方案。")
else:
    print("正常日和停运日的最优方案不同。给定概率 p 后，用 expected_orders(p).idxmax() 选择仓库。")

# 对照：若只关注影响最大的仓库停运当天，哪些选址能够处理最多订单？
critical_plant = disruptions.loc[disruptions["类型"].eq("仓库"), "停运对象"].iloc[0]
stress_day = plan_throughput[critical_plant]
best_stress = " 或 ".join(stress_day.index[stress_day.eq(stress_day.max())])
print(f"仅看 {critical_plant} 停运当天，{best_stress} 均可处理 {stress_day.max():,} 笔。"
      f"第二问的 {sites_text} 组合可处理 {stress_day[sites_text]:,} 笔。")

共比较 191 个方案。下表列出原容量与停运日平均处理量最高的三个扩容方案。


,正常日处理笔数,停运日平均处理笔数,较原容量的停运日平均增量
扩容仓库,,,
原容量,2291,2185.89,0.00
PLANT02、PLANT03,2491,2378.75,192.86
PLANT02、PLANT12,2456,2344.18,158.29
PLANT03、PLANT12,2456,2344.18,158.29


PLANT02、PLANT03 在正常日和停运日平均都居首：2,491 笔和 2,378.75 笔。在各停运情景等可能的假设下，任意单日停运概率都可选这个方案。
仅看 PLANT03 停运当天，PLANT02、PLANT10 或 PLANT02、PLANT12 均可处理 1,443 笔。第二问的 PLANT02、PLANT03 组合可处理 1,378 笔。
